# Owner Assets vs Contract History Comparison

This notebook finds properties that appear in contract history with an active, pending, or termination-request contract, but are missing from the expected owner-assets list.

Expected owner-assets presence:

- Contract history `OwnerContracts`: property should exist in `owner-assets/owned` or `owner-assets/leased`.
- Contract history `TenantContracts`: property should exist in `owner-assets/rented`.

Outputs are written under `runs/owner_assets_contract_history_comparison_<timestamp>`.


In [ ]:
import base64
import csv
import json
import os
import re
from datetime import datetime
from pathlib import Path

import requests

try:
    from google.colab import userdata
except Exception:
    userdata = None

DDA_OWNER_EMIRATES_IDS = [
    "784199515347708",
    "784199668638416",
    "784197265140323",
    "784195279540512",
]
PERSONAL_OWNER_EMIRATES_ID = "784198721116758"
OWNER_EMIRATES_IDS = DDA_OWNER_EMIRATES_IDS + [PERSONAL_OWNER_EMIRATES_ID]

PROPERTY_TYPE_IDS = (2, 3)
OWNER_ASSET_LIST_TYPES = ("owned", "leased", "rented")
EXPECTED_LIST_TYPES_BY_HISTORY_ROLE = {
    "owner": ("owned", "leased"),
    "tenant": ("rented",),
}
CONTRACT_HISTORY_SOURCE = "actual"  # "actual" or "proxy"
REQUEST_TIMEOUT_SECONDS = 90
CURRENT_BEARER_TOKEN = None


def load_local_env(path=".env"):
    env_path = Path(path)
    if not env_path.exists():
        return
    with env_path.open("r", encoding="utf-8") as f:
        for raw_line in f:
            line = raw_line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            key, value = line.split("=", 1)
            os.environ.setdefault(key.strip(), value.strip().strip('"').strip("'"))


load_local_env()


def get_secret(name, default=None):
    if userdata is not None:
        value = userdata.get(name)
        if value is not None:
            return value
    return os.environ.get(name, default)


def require_secret(name):
    value = get_secret(name)
    if not value:
        raise RuntimeError(f"Missing required secret/env var: {name}")
    return value


BASIC_AUTH = require_secret("BASIC_AUTH")
CONSUMER_ID = require_secret("CONSUMER_ID")
DLD_BASE_URL = require_secret("DLD_BASE_URL").rstrip("/")
DLD_PROXY_URL = (get_secret("DLD_PROXY_URL", "") or "").rstrip("/")
IDS_BASE_URL = require_secret("IDS_BASE_URL")

print("Configured Emirates IDs:")
for eid in OWNER_EMIRATES_IDS:
    print("-", eid)


In [ ]:
class ApiRequestError(Exception):
    def __init__(self, message, *, url=None, status_code=None, response=None):
        super().__init__(message)
        self.url = url
        self.status_code = status_code
        self.response = response


def decode_json_like(value):
    if isinstance(value, str):
        text = value.strip()
        if text.startswith("{") or text.startswith("["):
            try:
                return decode_json_like(json.loads(text))
            except Exception:
                return value
        return value
    if isinstance(value, dict):
        return {key: decode_json_like(inner_value) for key, inner_value in value.items()}
    if isinstance(value, list):
        return [decode_json_like(item) for item in value]
    return value


def response_payload(response):
    try:
        return decode_json_like(response.json())
    except Exception:
        return decode_json_like(response.text)


def compact_payload(value, max_chars=1500):
    if isinstance(value, str):
        text = value
    else:
        text = json.dumps(value, ensure_ascii=False, default=str)
    return text if len(text) <= max_chars else text[:max_chars] + "..."


def get_bearer_token():
    url = IDS_BASE_URL
    headers = {
        "Authorization": f"Basic {BASIC_AUTH}",
        "Content-Type": "application/x-www-form-urlencoded",
    }
    response = requests.post(url, headers=headers, data={}, timeout=REQUEST_TIMEOUT_SECONDS)
    data = response_payload(response)
    if response.status_code >= 400:
        raise ApiRequestError(
            f"iPaaS bearer token failed HTTP {response.status_code}: {compact_payload(data)}",
            url=url,
            status_code=response.status_code,
            response=data,
        )
    token = data.get("id_token") if isinstance(data, dict) else None
    if not token:
        raise ApiRequestError(f"iPaaS bearer token response has no id_token: {compact_payload(data)}", url=url, response=data)

    global CURRENT_BEARER_TOKEN
    CURRENT_BEARER_TOKEN = token
    return token


def get_active_bearer_token(force_refresh=False):
    global CURRENT_BEARER_TOKEN
    if force_refresh or not CURRENT_BEARER_TOKEN:
        return get_bearer_token()
    return CURRENT_BEARER_TOKEN


def request_with_bearer(method, url, headers=None, retry_on_401=True, **kwargs):
    request_headers = dict(headers or {})
    request_headers["Authorization"] = f"Bearer {get_active_bearer_token()}"
    kwargs.setdefault("timeout", REQUEST_TIMEOUT_SECONDS)
    response = requests.request(method, url, headers=request_headers, **kwargs)

    if response.status_code == 401 and retry_on_401:
        print("iPaaS bearer token expired; refreshing and retrying request once.")
        request_headers["Authorization"] = f"Bearer {get_active_bearer_token(force_refresh=True)}"
        response = requests.request(method, url, headers=request_headers, **kwargs)

    return response


def api_errors(payload):
    if not isinstance(payload, dict):
        return []
    errors = []
    response_code = payload.get("ResponseCode") or payload.get("responseCode")
    if response_code is not None:
        normalized_response_code = str(response_code).strip().lower()
        if normalized_response_code not in {"success", "200", "ok"}:
            errors.append(f"ResponseCode={response_code}")
    for key in ("ErrorMessage", "errorMessage", "Message", "message"):
        value = payload.get(key)
        if value:
            errors.append(str(value))
    for error in payload.get("ValidationErrorsList") or payload.get("validationErrorsList") or []:
        if not isinstance(error, dict):
            errors.append(str(error))
            continue
        error_number = error.get("ErrorNumber") or error.get("errorNumber")
        message_set = error.get("ErrorMessageSet") or error.get("errorMessageSet") or {}
        message = error.get("ErrorMessage") or error.get("errorMessage")
        if isinstance(message_set, dict):
            message = message or message_set.get("EnglishName") or message_set.get("englishName") or message_set.get("ArabicName")
        normalized_message = str(message or "").strip().lower()
        if error_number not in (None, 0, "0"):
            errors.append(f"ErrorNumber={error_number}; {message or compact_payload(error)}")
        elif message and normalized_message not in {"no errors found", "no errors found."}:
            errors.append(str(message))
    return errors


def assert_ok_response(response, api_name, url):
    data = response_payload(response)
    if response.status_code >= 400:
        raise ApiRequestError(
            f"{api_name} failed HTTP {response.status_code}: {compact_payload(data)}",
            url=url,
            status_code=response.status_code,
            response=data,
        )
    errors = api_errors(data)
    if errors:
        raise ApiRequestError(f"{api_name} returned API errors: {' | '.join(errors)}", url=url, status_code=response.status_code, response=data)
    return data


def dld_headers(dld_token, *, accept_text=False):
    headers = {
        "Token": dld_token,
        "consumer-id": CONSUMER_ID,
    }
    if accept_text:
        headers["accept"] = "text/plain"
    return headers


def get_dld_token(emirates_id):
    bearer_token = get_active_bearer_token()
    url = f"{DLD_BASE_URL}/generaltokenservice/1.0.0/auth"
    auth_obj = {
        "Method": "EmiratesId",
        "MethodIdentity": str(emirates_id),
        "MethodPasscode": "",
        "DeviceKey": "MyPC",
        "ApplicationKey": "DubaiNow",
        "Platform": "Mobile",
    }
    encoded = base64.b64encode(json.dumps(auth_obj).encode()).decode()
    headers = {
        "consumer-id": CONSUMER_ID,
        "x-nv-header": encoded,
        "Content-Type": "application/json",
        "Authorization": f"Bearer {bearer_token}",
    }
    response = request_with_bearer("post", url, headers=headers)
    data = assert_ok_response(response, "DLD token", url)
    token = data.get("token") if isinstance(data, dict) else None
    if not token:
        raise ApiRequestError(f"DLD token response has no token: {compact_payload(data)}", url=url, response=data)
    return token


In [ ]:
def display_name(value):
    if isinstance(value, dict):
        return value.get("EnglishName") or value.get("englishName") or value.get("ArabicName") or value.get("arabicName") or value.get("Value") or value.get("Name")
    return value


def nested_get(value, *keys):
    current = value
    for key in keys:
        if isinstance(current, dict):
            current = current.get(key)
        elif isinstance(current, list) and isinstance(key, int) and len(current) > key:
            current = current[key]
        else:
            return None
    return current


def first_present(*values):
    for value in values:
        if value is None:
            continue
        if isinstance(value, str) and not value.strip():
            continue
        return value
    return None


def compact_key(value):
    if value is None:
        return None
    text = str(value).strip()
    return text or None


def normalize_text(value):
    text = str(value or "").strip().lower()
    return re.sub(r"[^a-z0-9]+", "", text)


def normalize_status(value):
    return normalize_text(display_name(value))


def contract_status_values(contract):
    status = contract.get("ContractStatus") or contract.get("Status") if isinstance(contract, dict) else None
    return [
        display_name(status),
        nested_get(status, "Identity"),
        nested_get(status, "Value"),
        contract.get("ContractStatusName") if isinstance(contract, dict) else None,
        contract.get("StatusName") if isinstance(contract, dict) else None,
        contract.get("ContractStatus") if isinstance(contract, dict) and not isinstance(contract.get("ContractStatus"), dict) else None,
        contract.get("Status") if isinstance(contract, dict) and not isinstance(contract.get("Status"), dict) else None,
    ]


def contract_status_text(contract):
    for value in contract_status_values(contract):
        text = compact_key(display_name(value))
        if text:
            return text
    return None


def is_in_scope_contract_status(contract):
    normalized_values = [normalize_status(value) for value in contract_status_values(contract)]
    normalized_values = [value for value in normalized_values if value]
    for value in normalized_values:
        if value in {"active", "pending"}:
            return True
        if "termin" in value and "request" in value:
            return True
    return False


def property_id_from(item):
    if not isinstance(item, dict):
        return None
    return first_present(
        item.get("PropertyId"),
        item.get("PropertyID"),
        item.get("PROPERTY_ID"),
        nested_get(item, "Property", "PropertyId"),
        nested_get(item, "Property", "PropertyID"),
    )


def property_row_value_from_item(item):
    if not isinstance(item, dict):
        return None
    return first_present(
        item.get("PropertyRowValue"),
        item.get("propertyRowValue"),
        item.get("RowValue"),
        nested_get(item, "Property", "RowValue"),
        nested_get(item, "Property", "PropertyRowValue"),
    )


def contract_row_value_from_contract(contract):
    if not isinstance(contract, dict):
        return None
    return first_present(
        contract.get("AssociatedContractRowValue"),
        contract.get("ContractRowValue"),
        contract.get("DataRowValue"),
        nested_get(contract, "AssociatedTenancyContract", "RowValue"),
    )


def contract_number_from_contract(contract):
    if not isinstance(contract, dict):
        return None
    return first_present(
        contract.get("ContractNumber"),
        nested_get(contract, "AssociatedTenancyContract", "ContractNumber"),
    )


def property_type_id_from_item(item):
    if not isinstance(item, dict):
        return None
    return first_present(
        item.get("PropertyTypeId"),
        item.get("PropertyTypeID"),
        item.get("PROPERTY_TYPE_ID"),
        nested_get(item, "PropertyType", "Identity"),
        nested_get(item, "PropertyType", "Value"),
    )


def property_title_from_item(item):
    if not isinstance(item, dict):
        return None
    return display_name(first_present(
        item.get("Title"),
        item.get("PropertyName"),
        item.get("PropertyNumber"),
        item.get("Name"),
        nested_get(item, "Property", "Title"),
        nested_get(item, "Property", "PropertyName"),
        nested_get(item, "Property", "PropertyNumber"),
    ))


def owner_asset_keys(item):
    keys = []
    property_id = compact_key(property_id_from(item))
    property_row_value = compact_key(property_row_value_from_item(item))
    title = normalize_text(property_title_from_item(item))
    if property_id:
        keys.append(("property_id", property_id))
    if property_row_value:
        keys.append(("property_row_value", property_row_value))
    if title:
        keys.append(("property_title", title))
    return keys


In [ ]:
def get_properties_list(dld_token, list_type, property_type_id):
    url = f"{DLD_BASE_URL}/generalservices/1.0.0/owner-assets/{list_type}/{property_type_id}"
    response = request_with_bearer("get", url, headers=dld_headers(dld_token))
    data = assert_ok_response(response, f"Properties list {list_type}/{property_type_id}", url)
    properties = nested_get(data, "Response", "Properties")
    if not isinstance(properties, list):
        raise ApiRequestError(f"Properties list {list_type}/{property_type_id} returned no Response.Properties list", url=url, response=data)
    return properties, data, url


def get_contract_history(dld_token, source=CONTRACT_HISTORY_SOURCE):
    if source == "proxy":
        if not DLD_PROXY_URL:
            raise RuntimeError("DLD_PROXY_URL is required when CONTRACT_HISTORY_SOURCE='proxy'")
        url = f"{DLD_PROXY_URL}/ejariservices/properties/getcontracts"
    else:
        url = f"{DLD_BASE_URL}/dxbnwejaricontracts/1.0.0/properties/getcontracts"
    response = request_with_bearer("get", url, headers=dld_headers(dld_token, accept_text=True))
    data = assert_ok_response(response, f"Contract history ({source})", url)
    response_data = data.get("Response") if isinstance(data, dict) else {}
    owner_contracts = response_data.get("OwnerContracts") if isinstance(response_data, dict) else []
    tenant_contracts = response_data.get("TenantContracts") if isinstance(response_data, dict) else []
    owner_contracts = owner_contracts if isinstance(owner_contracts, list) else []
    tenant_contracts = tenant_contracts if isinstance(tenant_contracts, list) else []

    contracts = []
    for contract in owner_contracts:
        if isinstance(contract, dict):
            item = dict(contract)
            item["_history_role"] = "owner"
            item["_history_group"] = "OwnerContracts"
            contracts.append(item)
    for contract in tenant_contracts:
        if isinstance(contract, dict):
            item = dict(contract)
            item["_history_role"] = "tenant"
            item["_history_group"] = "TenantContracts"
            contracts.append(item)
    return contracts, data, url


def fetch_owner_assets(dld_token):
    all_assets = []
    counts = {}
    errors = []
    for list_type in OWNER_ASSET_LIST_TYPES:
        for property_type_id in PROPERTY_TYPE_IDS:
            key = f"{list_type}/{property_type_id}"
            try:
                properties, _, url = get_properties_list(dld_token, list_type, property_type_id)
                counts[key] = len(properties)
                for prop in properties:
                    if not isinstance(prop, dict):
                        continue
                    item = dict(prop)
                    item["_owner_asset_list_type"] = list_type
                    item["_property_type_id"] = property_type_id
                    item["_owner_asset_list_url"] = url
                    all_assets.append(item)
            except Exception as exc:
                counts[key] = None
                errors.append({
                    "owner_asset_list": key,
                    "error": str(exc),
                    "url": getattr(exc, "url", None),
                    "status_code": getattr(exc, "status_code", None),
                    "response": compact_payload(getattr(exc, "response", None), 800),
                })
    return all_assets, counts, errors


def build_owner_assets_index(assets):
    index = {}
    for asset in assets:
        for bucket, key in owner_asset_keys(asset):
            index.setdefault(bucket, {}).setdefault(str(key), []).append(asset)
    return index


def find_owner_asset_matches(contract, owner_assets_index):
    matches = []
    seen = set()
    for bucket, key in owner_asset_keys(contract):
        for asset in owner_assets_index.get(bucket, {}).get(str(key), []):
            marker = (
                asset.get("_owner_asset_list_type"),
                asset.get("_property_type_id"),
                compact_key(property_id_from(asset)),
                compact_key(property_row_value_from_item(asset)),
            )
            if marker in seen:
                continue
            seen.add(marker)
            asset_copy = dict(asset)
            asset_copy["_match_bucket"] = bucket
            asset_copy["_match_key"] = key
            matches.append(asset_copy)
    return matches


def join_unique(values):
    cleaned = []
    for value in values or []:
        if value is None:
            continue
        text = str(value).strip()
        if text and text not in cleaned:
            cleaned.append(text)
    return "; ".join(cleaned)


def build_comparison_row(emirates_id, contract, matches):
    role = contract.get("_history_role")
    expected_list_types = EXPECTED_LIST_TYPES_BY_HISTORY_ROLE.get(role, ())
    matched_lists = [match.get("_owner_asset_list_type") for match in matches]
    matching_expected = [match for match in matches if match.get("_owner_asset_list_type") in expected_list_types]
    return {
        "emirates_id": emirates_id,
        "history_group": contract.get("_history_group"),
        "history_role": role,
        "contract_status": contract_status_text(contract),
        "contract_number": contract_number_from_contract(contract),
        "contract_row_value": contract_row_value_from_contract(contract),
        "property_id": property_id_from(contract),
        "property_type_id": property_type_id_from_item(contract),
        "property_name": property_title_from_item(contract),
        "property_row_value": property_row_value_from_item(contract),
        "expected_owner_asset_lists": join_unique(expected_list_types),
        "present_in_expected_owner_asset_list": bool(matching_expected),
        "present_in_any_owner_asset_list": bool(matches),
        "matched_owner_asset_lists": join_unique(matched_lists),
        "matched_owner_asset_property_type_ids": join_unique(match.get("_property_type_id") for match in matches),
        "matched_by": join_unique(f"{match.get('_match_bucket')}={match.get('_match_key')}" for match in matches),
    }


def compare_history_to_owner_assets(emirates_id, contracts, owner_assets):
    owner_assets_index = build_owner_assets_index(owner_assets)
    in_scope_contracts = [contract for contract in contracts if is_in_scope_contract_status(contract)]
    all_rows = []
    missing_rows = []
    for contract in in_scope_contracts:
        matches = find_owner_asset_matches(contract, owner_assets_index)
        row = build_comparison_row(emirates_id, contract, matches)
        all_rows.append(row)
        if not row["present_in_expected_owner_asset_list"]:
            missing_rows.append(row)
    return all_rows, missing_rows


In [ ]:
OUTPUT_DIR = Path("runs") / f"owner_assets_contract_history_comparison_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

COMPARISON_COLUMNS = [
    "emirates_id",
    "history_group",
    "history_role",
    "contract_status",
    "contract_number",
    "contract_row_value",
    "property_id",
    "property_type_id",
    "property_name",
    "property_row_value",
    "expected_owner_asset_lists",
    "present_in_expected_owner_asset_list",
    "present_in_any_owner_asset_list",
    "matched_owner_asset_lists",
    "matched_owner_asset_property_type_ids",
    "matched_by",
]
SUMMARY_COLUMNS = [
    "emirates_id",
    "contract_history_total",
    "contract_history_in_scope",
    "missing_from_expected_owner_assets",
    "owned_2_count",
    "owned_3_count",
    "leased_2_count",
    "leased_3_count",
    "rented_2_count",
    "rented_3_count",
    "owner_asset_fetch_error_count",
]
ERROR_COLUMNS = ["emirates_id", "stage", "owner_asset_list", "error", "url", "status_code", "response"]

all_comparison_rows = []
missing_comparison_rows = []
summary_rows = []
error_rows = []


def write_csv(path, rows, columns):
    with Path(path).open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=columns, extrasaction="ignore")
        writer.writeheader()
        writer.writerows(rows)


def write_json(path, rows):
    Path(path).write_text(json.dumps(rows, ensure_ascii=False, indent=2, default=str), encoding="utf-8")


def save_outputs():
    write_csv(OUTPUT_DIR / "all_in_scope_contract_history_comparison.csv", all_comparison_rows, COMPARISON_COLUMNS)
    write_csv(OUTPUT_DIR / "missing_from_expected_owner_assets.csv", missing_comparison_rows, COMPARISON_COLUMNS)
    write_csv(OUTPUT_DIR / "summary.csv", summary_rows, SUMMARY_COLUMNS)
    write_csv(OUTPUT_DIR / "errors.csv", error_rows, ERROR_COLUMNS)
    write_json(OUTPUT_DIR / "all_in_scope_contract_history_comparison.json", all_comparison_rows)
    write_json(OUTPUT_DIR / "missing_from_expected_owner_assets.json", missing_comparison_rows)
    write_json(OUTPUT_DIR / "summary.json", summary_rows)
    write_json(OUTPUT_DIR / "errors.json", error_rows)


print("Output directory:", OUTPUT_DIR)


In [ ]:
def should_process_emirates_id(emirates_id, process_all):
    if process_all:
        return True
    response = input(f"Process Emirates ID {emirates_id}? (yes/no) [yes]: ").strip().lower()
    if response in {"", "y", "yes"}:
        return True
    print(f"Skipped Emirates ID {emirates_id}")
    return False


process_all_response = input("Process all configured Emirates IDs? (yes/no) [yes]: ").strip().lower()
process_all = process_all_response in {"", "y", "yes", "all"}

for emirates_id in OWNER_EMIRATES_IDS:
    if not should_process_emirates_id(emirates_id, process_all):
        continue

    print(f"\nProcessing Emirates ID {emirates_id}")
    try:
        # Start each owner with a fresh iPaaS bearer token. request_with_bearer still refreshes once on 401 during the run.
        CURRENT_BEARER_TOKEN = None
        dld_token = get_dld_token(emirates_id)

        owner_assets, owner_asset_counts, owner_asset_errors = fetch_owner_assets(dld_token)
        for error in owner_asset_errors:
            error_rows.append({"emirates_id": emirates_id, "stage": "owner_assets_fetch", **error})

        contracts, _, contract_history_url = get_contract_history(dld_token)
        all_rows, missing_rows = compare_history_to_owner_assets(emirates_id, contracts, owner_assets)

        all_comparison_rows.extend(all_rows)
        missing_comparison_rows.extend(missing_rows)
        summary_rows.append({
            "emirates_id": emirates_id,
            "contract_history_total": len(contracts),
            "contract_history_in_scope": len(all_rows),
            "missing_from_expected_owner_assets": len(missing_rows),
            "owned_2_count": owner_asset_counts.get("owned/2"),
            "owned_3_count": owner_asset_counts.get("owned/3"),
            "leased_2_count": owner_asset_counts.get("leased/2"),
            "leased_3_count": owner_asset_counts.get("leased/3"),
            "rented_2_count": owner_asset_counts.get("rented/2"),
            "rented_3_count": owner_asset_counts.get("rented/3"),
            "owner_asset_fetch_error_count": len(owner_asset_errors),
        })

        print(f"Contract history rows: {len(contracts)}")
        print(f"In-scope active/pending/termination-request rows: {len(all_rows)}")
        print(f"Missing from expected owner-assets lists: {len(missing_rows)}")
        if owner_asset_errors:
            print(f"Owner-assets fetch errors: {len(owner_asset_errors)}")

    except Exception as exc:
        print(f"ERROR processing Emirates ID {emirates_id}: {exc}")
        error_rows.append({
            "emirates_id": emirates_id,
            "stage": "emirates_id_processing",
            "owner_asset_list": None,
            "error": str(exc),
            "url": getattr(exc, "url", None),
            "status_code": getattr(exc, "status_code", None),
            "response": compact_payload(getattr(exc, "response", None), 800),
        })
    finally:
        save_outputs()
        print("Saved current output files.")

print("\nDone.")
print("Output directory:", OUTPUT_DIR)
print("Total in-scope contract-history rows:", len(all_comparison_rows))
print("Total missing rows:", len(missing_comparison_rows))


In [ ]:
try:
    import pandas as pd
    if missing_comparison_rows:
        display(pd.DataFrame(missing_comparison_rows, columns=COMPARISON_COLUMNS))
    else:
        print("No missing properties found in the processed data.")
    display(pd.DataFrame(summary_rows, columns=SUMMARY_COLUMNS))
except Exception:
    print("Missing rows preview:")
    print(json.dumps(missing_comparison_rows[:20], ensure_ascii=False, indent=2, default=str))
    print("Summary:")
    print(json.dumps(summary_rows, ensure_ascii=False, indent=2, default=str))
